### Load Data

In [47]:
from loader import DocumentLoader

CSV_FILE = '/home/coen-molyneaux/code/ml/nuclear-log-nn/dataset/log.csv'
DocumentLoader = DocumentLoader(csvfile=CSV_FILE)
x, y = DocumentLoader.load()

### Model Arch

In [48]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(9,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 64)             │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,753 (10.75 KB)

 Trainable params: 2,753 (10.75 KB)

 Non-trainable params: 0 (0.00 B)

### Training, Val, and Test Data

In [49]:
import numpy as np
import tensorflow as tf

x = np.asarray(x, dtype=np.float32)
y = np.asarray(y, dtype=np.float32).reshape(-1, 1)

N = len(x)
assert len(x) == len(y), "X and y length mismatch"

train_end = int(0.70 * N)
val_end   = int(0.85 * N)

x_train, y_train = x[:train_end], y[:train_end]
x_val,   y_val   = x[train_end:val_end], y[train_end:val_end]
x_test,  y_test  = x[val_end:], y[val_end:]

mean = x_train.mean(axis=0)
std  = x_train.std(axis=0) + 1e-8

x_train = (x_train - mean) / std
x_val   = (x_val   - mean) / std
x_test  = (x_test  - mean) / std

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(1000).batch(32)
val_ds   = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(32)

### Training

In [50]:
tf.keras.config.disable_traceback_filtering()
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop]
)

Epoch 1/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4215 - mae: 0.4458 - val_loss: 14434.7949 - val_mae: 111.5192
Epoch 2/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1086 - mae: 0.2573 - val_loss: 9670.4551 - val_mae: 91.2062
Epoch 3/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0801 - mae: 0.2284 - val_loss: 4555.1133 - val_mae: 62.2738
Epoch 4/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0684 - mae: 0.2119 - val_loss: 3435.4832 - val_mae: 54.0810
Epoch 5/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0630 - mae: 0.2045 - val_loss: 1517.9696 - val_mae: 35.6769
Epoch 6/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0603 - mae: 0.2001 - val_loss: 1368.6112 - val_mae: 33.7585
Epoch 7/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0558 - mae: 0.1927 - val_loss: 866.3419 - val_mae: 26.8013
Epoch 8/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0552 - mae: 0.1909 - val_loss: 573.0299 - val_mae: 21.6215
Epoch 9/50
192/192 ━━━━━

In [51]:
model.save("model.keras")